# AkStudioFreeai — Free Personal Voice Training

This notebook prepares a personal RVC voice-conversion training environment.

**Use only your own voice or a voice you have explicit permission to use.**

Requirements:
- Google Colab account
- Runtime with an available NVIDIA GPU
- A ZIP exported from AkStudioFreeai Voice Dataset
- Approximately 10–50 minutes of clean recordings is a useful training target

Free Colab GPU access is limited and not guaranteed. Sessions can disconnect. Save your dataset and trained model outside the temporary runtime.

RVC performs voice conversion. It is not a standalone text-to-singing generator.

In [ ]:
# STEP 1 — Check the free runtime
import sys, subprocess
print("Python:", sys.version)
gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else "No NVIDIA GPU detected.")
if gpu.returncode != 0:
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, if available, and rerun.")
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("This RVC project branch targets Python 3.12. Use a compatible Colab runtime.")

In [ ]:
# STEP 2 — Install system tools and clone official RVC
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!git clone --depth 1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/RVC
%cd /content/RVC
!git rev-parse --short HEAD
!python --version

## STEP 3 — Install RVC dependencies

The official RVC documentation currently describes Python 3.12 and separate dependency sets for GPU generations. The next cell targets older NVIDIA GPUs using CUDA 11.8, such as a T4. If your runtime has a different GPU, stop and follow the official installation guide rather than forcing these packages.

In [ ]:
# STEP 3 — CUDA 11.8 dependency setup (T4 / older NVIDIA GPU)
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install torch==2.7.1+cu118 torchaudio==2.7.1+cu118 --index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://pypi.org/simple
!python -m pip install -r requirments_cu118_py312.txt
!python -m pip install --upgrade huggingface_hub
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Do not continue with GPU training.")

## STEP 4 — Download official RVC model assets

These are the pretrained feature extraction and training assets referenced by the RVC project. Downloads may take time.

In [ ]:
# STEP 4 — Official model assets
!hf download lj1995/VoiceConversionWebUI --revision main --include "hubert_base/*" --local-dir assets
!hf download lj1995/VoiceConversionWebUI rmvpe.pt --revision main --local-dir assets/rmvpe
!hf download lj1995/VoiceConversionWebUI --revision main --include "pretrained/*" "pretrained_v2/*" --local-dir assets
!hf download lj1995/VoiceConversionWebUI mute.zip --revision main --local-dir .model-downloads
!python -m zipfile -e .model-downloads/mute.zip logs

## STEP 5 — Upload your AkStudioFreeai dataset ZIP

On your website, open **Voice Dataset → Export Dataset ZIP**. Upload that ZIP in the next cell.

The notebook converts supported audio files to mono WAV files at 40 kHz. Review the output and remove noisy, clipped, or unrelated samples before training.

In [ ]:
# STEP 5 — Upload and prepare your own recordings
from google.colab import files
from pathlib import Path
import zipfile, shutil, subprocess

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_names:
    raise RuntimeError("Upload the AkStudioFreeai Voice Dataset ZIP file.")

source_zip = Path("/content") / zip_names[0]
extract_dir = Path("/content/akstudio_voice_upload")
shutil.rmtree(extract_dir, ignore_errors=True)
extract_dir.mkdir(parents=True)

with zipfile.ZipFile(source_zip) as z:
    z.extractall(extract_dir)

extensions = {".wav", ".mp3", ".m4a", ".webm", ".ogg", ".flac", ".aac"}
audio_files = [p for p in extract_dir.rglob("*") if p.is_file() and p.suffix.lower() in extensions]
if not audio_files:
    raise RuntimeError("No supported audio recordings found in the ZIP.")

dataset_dir = Path("/content/akstudio_rvc_dataset")
shutil.rmtree(dataset_dir, ignore_errors=True)
dataset_dir.mkdir()

converted = []
for i, audio in enumerate(audio_files, 1):
    target = dataset_dir / f"sample_{i:03d}.wav"
    result = subprocess.run([
        "ffmpeg", "-y", "-i", str(audio), "-vn",
        "-ac", "1", "-ar", "40000", str(target)
    ], capture_output=True, text=True)
    if result.returncode == 0 and target.exists() and target.stat().st_size > 10000:
        converted.append(target)

print("Audio files found:", len(audio_files))
print("WAV files prepared:", len(converted))
print("Dataset folder:", dataset_dir)
if not converted:
    raise RuntimeError("No audio files could be converted. Check your ZIP recordings.")

In [ ]:
# STEP 6 — Download a backup copy of the prepared WAV dataset
import shutil
prepared_zip = shutil.make_archive(
    "/content/AkStudioFreeai-RVC-WAV-Dataset",
    "zip",
    root_dir="/content",
    base_dir="akstudio_rvc_dataset"
)
print("Prepared dataset:", prepared_zip)
files.download(prepared_zip)

## STEP 7 — Launch RVC WebUI for training

The official project provides its training and inference WebUI on port 7865. This cell attempts to launch it with a temporary public Gradio link.

**Important:** Colab may terminate sessions running web interfaces or long-running jobs. This is an experimental free workflow, not a guaranteed hosting service. Keep your notebook open, save checkpoints frequently, and never rely on the temporary runtime as permanent storage.

In [ ]:
# STEP 7 — Launch the official RVC training interface
import subprocess, time, re
from pathlib import Path

log_path = "/content/rvc-webui.log"
log_file = open(log_path, "w")
process = subprocess.Popen(
    ["python", "webui.py", "--noautoopen"],
    cwd="/content/RVC",
    stdout=log_file,
    stderr=subprocess.STDOUT
)

share_url = None
for _ in range(90):
    time.sleep(5)
    text = Path(log_path).read_text(errors="ignore")
    match = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", text)
    if match:
        share_url = match.group(0)
        break
    if process.poll() is not None:
        break

if share_url:
    print("RVC WebUI:", share_url)
    print("Open this temporary link to configure and train your own voice.")
else:
    print("WebUI link not detected. Recent log:")
    print(Path(log_path).read_text(errors="ignore")[-5000:])
    print("Process status:", process.poll())

## Training guidance

1. In RVC WebUI, create a new experiment/model name.
2. Set the training dataset folder to `/content/akstudio_rvc_dataset`.
3. Follow the WebUI preprocessing, feature extraction, and training steps.
4. Start with conservative training settings and listen to test conversions.
5. Export/download the trained model checkpoint and index before the Colab session ends.

This notebook does not guarantee successful training. If a dependency, GPU, or WebUI step fails, keep the error output and use the official RVC documentation.

Official project: https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI
Google Colab FAQ: https://research.google.com/colaboratory/faq.html